# Top 10 Aadhaar Insights Analysis

This notebook generates 10 actionable insights from the cleaned UIDAI datasets. Each insight is designed to identify specific patterns, anomalies, or trends in Aadhaar enrolment and update activities across India.

**Methodology:**
1.  **Load Cleaned Data**: Ingest the final, validated datasets from the `final_pipeline_output` directory.
2.  **Pre-process & Merge**: Standardize data types, handle dates, and create a unified DataFrame.
3.  **Generate Insights**: For each of the 10 points, calculate the required metrics and identify the districts or states that match the criteria.
4.  **Visualize & Report**: Display the findings in clear, interpretable tables.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import warnings

# --- Configuration ---
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 100)
ROOT = Path(os.getcwd()).resolve()
DATA_DIR = ROOT / "final_pipeline_output"
LOGS_DIR = DATA_DIR / "logs"

print(f"Project Root: {ROOT}")
print(f"Data Directory: {DATA_DIR}")

Project Root: D:\exp\uidai-datathon-2026-participation
Data Directory: D:\exp\uidai-datathon-2026-participation\final_pipeline_output


## 1. Load and Pre-process Data

This section loads the cleaned biometric, demographic, and enrolment CSVs, concatenates them into single DataFrames, and then merges them into a master analysis table.

In [2]:
def load_data_from_category(category_path: Path) -> pd.DataFrame:
    """Loads and concatenates all CSV files from a given category directory."""
    if not category_path.exists():
        print(f"[WARN] Directory not found: {category_path}")
        return pd.DataFrame()
    
    csv_files = list(category_path.glob("*.csv"))
    if not csv_files:
        print(f"[WARN] No CSV files found in {category_path}")
        return pd.DataFrame()
        
    dfs = [pd.read_csv(file, dtype=str) for file in csv_files]
    return pd.concat(dfs, ignore_index=True)

# Load the data
print("Loading datasets...")
enrolment_df = load_data_from_category(DATA_DIR / "enrolment")
biometric_df = load_data_from_category(DATA_DIR / "biometric")
demographic_df = load_data_from_category(DATA_DIR / "demographic")

# --- Pre-processing ---
print("Pre-processing data...")

# Combine into a single master DataFrame
dataframes = {}
if not enrolment_df.empty:
    dataframes['enrolment'] = enrolment_df
if not biometric_df.empty:
    dataframes['biometric'] = biometric_df
if not demographic_df.empty:
    dataframes['demographic'] = demographic_df

# Convert numeric and date columns
for name, df in dataframes.items():
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    
    # Dynamically find numeric columns
    for col in df.columns:
        if 'age' in col or 'bio' in col or 'demo' in col:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Create total columns
if 'enrolment' in dataframes:
    enrolment_df['total_enrolment'] = enrolment_df['age_0_5'] + enrolment_df['age_5_17'] + enrolment_df['age_18_greater']
if 'biometric' in dataframes:
    biometric_df['total_biometric_updates'] = biometric_df['bio_age_5_17'] + biometric_df['bio_age_17_']
if 'demographic' in dataframes:
    demographic_df['total_demographic_updates'] = demographic_df['demo_age_5_17'] + demographic_df['demo_age_17_']

# Group by date and location to create a unified table
group_cols = ['date', 'state', 'district', 'pincode']
enrol_agg = enrolment_df.groupby(group_cols)['total_enrolment'].sum().reset_index() if 'enrolment' in dataframes else pd.DataFrame(columns=group_cols)
bio_agg = biometric_df.groupby(group_cols)['total_biometric_updates'].sum().reset_index() if 'biometric' in dataframes else pd.DataFrame(columns=group_cols)
demo_agg = demographic_df.groupby(group_cols)['total_demographic_updates'].sum().reset_index() if 'demographic' in dataframes else pd.DataFrame(columns=group_cols)

# Merge into a single master DataFrame
print("Merging dataframes...")
master_df = pd.merge(enrol_agg, bio_agg, on=group_cols, how='outer')
master_df = pd.merge(master_df, demo_agg, on=group_cols, how='outer')

# Fill NaNs with 0 after merging
for col in ['total_enrolment', 'total_biometric_updates', 'total_demographic_updates']:
    if col in master_df.columns:
        master_df[col] = master_df[col].fillna(0)
    else:
        master_df[col] = 0

master_df['total_updates'] = master_df['total_biometric_updates'] + master_df['total_demographic_updates']
master_df = master_df.dropna(subset=['date', 'state', 'district'])

print("Master DataFrame created successfully.")
display(master_df.head())
print(f"Total records in master table: {len(master_df):,}")

Loading datasets...
Pre-processing data...
Merging dataframes...
Master DataFrame created successfully.


,date,state,district,pincode,total_enrolment,total_biometric_updates,total_demographic_updates,total_updates
0,2025-01-03,Andaman And Nicobar Islands,North And Middle Andaman,744201,0.0,81.0,0.0,81.0
1,2025-01-03,Andaman And Nicobar Islands,North And Middle Andaman,744202,0.0,298.0,422.0,720.0
2,2025-01-03,Andaman And Nicobar Islands,North And Middle Andaman,744204,0.0,179.0,0.0,179.0
3,2025-01-03,Andaman And Nicobar Islands,North And Middle Andaman,744205,0.0,191.0,0.0,191.0
4,2025-01-03,Andaman And Nicobar Islands,North And Middle Andaman,744209,0.0,34.0,0.0,34.0


Total records in master table: 805,891


### Insight 1: Aadhaar Access Desert Index
**What**: Districts where Aadhaar activity is structurally low, indicating potential infrastructure or access inequity.

**How**: We identify districts that have both a low number of active enrolment days and a daily average enrolment count that falls below the 25th percentile for their state.

In [3]:
# --- 1. Aadhaar Access Desert Index ---
print("--- Insight 1: Aadhaar Access Desert Index ---")

# Calculate total days in the dataset
total_days = (master_df['date'].max() - master_df['date'].min()).days + 1

# Group by district to get activity metrics
district_activity = master_df.groupby(['state', 'district']).agg(
    active_days=('date', 'nunique'),
    total_enrolments=('total_enrolment', 'sum')
).reset_index()

district_activity['avg_daily_enrolment'] = district_activity['total_enrolments'] / district_activity['active_days']

# Calculate state-level 25th percentile for average daily enrolment
state_percentiles = district_activity.groupby('state')['avg_daily_enrolment'].quantile(0.25).reset_index()
state_percentiles.rename(columns={'avg_daily_enrolment': 'state_25th_percentile'}, inplace=True)

# Merge percentiles back to district data
district_activity = pd.merge(district_activity, state_percentiles, on='state', how='left')

# Identify desert districts
desert_districts = district_activity[
    (district_activity['active_days'] < 0.3 * total_days) &
    (district_activity['avg_daily_enrolment'] < district_activity['state_25th_percentile'])
]

print(f"Found {len(desert_districts)} districts classified as 'Aadhaar Access Deserts'.")
display(desert_districts.sort_values('active_days').head(10))

--- Insight 1: Aadhaar Access Desert Index ---
Found 183 districts classified as 'Aadhaar Access Deserts'.


,state,district,active_days,total_enrolments,avg_daily_enrolment,state_25th_percentile
0,Andaman And Nicobar Islands,Nicobars,5,1.0,0.200000,0.448780
493,Rajasthan,Balotra,12,0.0,0.000000,40.585366
497,Rajasthan,Beawar,12,0.0,0.000000,40.585366
520,Rajasthan,Phalodi,13,0.0,0.000000,40.585366
417,Mizoram,Khawzawl,15,9.0,0.600000,0.614634
659,Uttar Pradesh,Mahrajganj,16,1.0,0.062500,75.378049
416,Mizoram,Hnahthial,17,0.0,0.000000,0.614634
528,Sikkim,Mangan,18,1.0,0.055556,0.090054
33,Arunachal Pradesh,Leparada,18,1.0,0.055556,0.447368
397,Manipur,Pherzawl,18,1.0,0.055556,4.987805


### Insight 2: Update Burden Ratio (Hidden Congestion)
**What**: Districts where the volume of updates is disproportionately high compared to new enrolments, suggesting friction in the update process.

**How**: We calculate an `update_ratio` (updates / enrolments) for each district and flag those where this ratio is significantly higher than their state's median.

In [4]:
# --- 2. Update Burden Ratio ---
print("\\n--- Insight 2: Update Burden Ratio ---")

district_totals = master_df.groupby(['state', 'district']).agg(
    total_enrolments=('total_enrolment', 'sum'),
    total_updates=('total_updates', 'sum')
).reset_index()

# Calculate update ratio, adding 1 to denominator to avoid division by zero
district_totals['update_ratio'] = district_totals['total_updates'] / (district_totals['total_enrolments'] + 1)

# Get the median update ratio for each state
state_median_ratio = district_totals.groupby('state')['update_ratio'].median().reset_index()
state_median_ratio.rename(columns={'update_ratio': 'state_median_ratio'}, inplace=True)

# Merge and find districts with high burden
district_burden = pd.merge(district_totals, state_median_ratio, on='state', how='left')
high_burden_districts = district_burden[district_burden['update_ratio'] > 2 * district_burden['state_median_ratio']] # Flag if more than 2x the median

print(f"Found {len(high_burden_districts)} districts with a high update burden.")
display(high_burden_districts.sort_values('update_ratio', ascending=False).head(10))

\n--- Insight 2: Update Burden Ratio ---
Found 44 districts with a high update burden.


,state,district,total_enrolments,total_updates,update_ratio,state_median_ratio
421,Mizoram,Mamit,27.0,11209.0,400.321429,40.747881
599,The Dadra And Nagar Haveli And Daman And Diu,Daman,31.0,6646.0,207.687500,99.176471
400,Manipur,Thoubal,370.0,75226.0,202.765499,70.668101
393,Manipur,Imphal East,366.0,71928.0,195.989101,70.668101
387,Maharashtra,Wardha,569.0,108865.0,190.991228,72.241624
423,Mizoram,Serchhip,49.0,9183.0,183.660000,40.747881
359,Maharashtra,Bhandara,543.0,96422.0,177.246324,72.241624
497,Rajasthan,Beawar,0.0,175.0,175.000000,42.294672
365,Maharashtra,Gadchiroli,730.0,127635.0,174.603283,72.241624
394,Manipur,Imphal West,474.0,82274.0,173.208421,70.668101


### Insight 3: Peak-Day Collapse Risk
**What**: Districts that experience massive, concentrated spikes in enrolment on a few days, indicating a risk of system collapse under pressure.

**How**: We calculate a `peak_ratio` (max daily enrolments / median daily enrolments). A ratio greater than 5 suggests extreme volatility and potential capacity issues.

In [5]:
# --- 3. Peak-Day Collapse Risk ---
print("\\n--- Insight 3: Peak-Day Collapse Risk ---")

daily_district_enrolments = master_df.groupby(['state', 'district', 'date'])['total_enrolment'].sum().reset_index()

# Calculate max and median daily enrolments for each district
peak_analysis = daily_district_enrolments.groupby(['state', 'district'])['total_enrolment'].agg(['max', 'median']).reset_index()
peak_analysis.rename(columns={'max': 'max_daily_enrolments', 'median': 'median_daily_enrolments'}, inplace=True)

# Calculate peak ratio, avoiding division by zero
peak_analysis['peak_ratio'] = peak_analysis['max_daily_enrolments'] / (peak_analysis['median_daily_enrolments'] + 1)

# Flag districts at risk
collapse_risk_districts = peak_analysis[peak_analysis['peak_ratio'] > 5]

print(f"Found {len(collapse_risk_districts)} districts with a high peak-day collapse risk.")
display(collapse_risk_districts.sort_values('peak_ratio', ascending=False).head(10))

\n--- Insight 3: Peak-Day Collapse Risk ---
Found 546 districts with a high peak-day collapse risk.


,state,district,max_daily_enrolments,median_daily_enrolments,peak_ratio
106,Bihar,Pashchim Champaran,9642.0,0.0,9642.000000
262,Karnataka,Bengaluru Rural,1654.0,0.0,1654.000000
406,Meghalaya,North Garo Hills,1495.0,0.0,1495.000000
146,Delhi,New Delhi,1091.0,0.0,1091.000000
182,Gujarat,Surendranagar,1310.0,0.5,873.333333
512,Rajasthan,Jalore,490.0,0.0,490.000000
355,Maharashtra,Ahilyanagar,368.0,0.0,368.000000
402,Meghalaya,East Garo Hills,1908.0,5.0,318.000000
409,Meghalaya,South West Garo Hills,1775.0,5.0,295.833333
413,Meghalaya,West Khasi Hills,4695.0,16.0,276.176471


### Insight 4: First-Time Enrolment Saturation Signal
**What**: Districts that may be approaching Aadhaar saturation, where the focus is shifting from new enrolments to updates.

**How**: We calculate the ratio of first-time enrolments to total activity (enrolments + updates). A consistently low ratio over time suggests saturation.

In [6]:
# --- 4. First-Time Enrolment Saturation Signal ---
print("\\n--- Insight 4: First-Time Enrolment Saturation Signal ---")

district_totals['first_time_ratio'] = district_totals['total_enrolments'] / (district_totals['total_enrolments'] + district_totals['total_updates'] + 1)

# Identify districts where updates significantly outweigh new enrolments
saturated_districts = district_totals[district_totals['first_time_ratio'] < 0.25] # Threshold: <25% of activity is new enrolments

print(f"Found {len(saturated_districts)} districts showing signs of enrolment saturation.")
display(saturated_districts.sort_values('first_time_ratio').head(10))

\n--- Insight 4: First-Time Enrolment Saturation Signal ---
Found 691 districts showing signs of enrolment saturation.


,state,district,total_enrolments,total_updates,update_ratio,first_time_ratio
493,Rajasthan,Balotra,0.0,148.0,148.000000,0.000000
520,Rajasthan,Phalodi,0.0,71.0,71.000000,0.000000
416,Mizoram,Hnahthial,0.0,39.0,39.000000,0.000000
40,Arunachal Pradesh,Pakke Kessang,0.0,87.0,87.000000,0.000000
213,Himachal Pradesh,Lahaul And Spiti,0.0,58.0,58.000000,0.000000
522,Rajasthan,Salumbar,0.0,92.0,92.000000,0.000000
497,Rajasthan,Beawar,0.0,175.0,175.000000,0.000000
421,Mizoram,Mamit,27.0,11209.0,400.321429,0.002403
599,The Dadra And Nagar Haveli And Daman And Diu,Daman,31.0,6646.0,207.687500,0.004642
400,Manipur,Thoubal,370.0,75226.0,202.765499,0.004894


### Insight 5: Female Access Gap
**What**: Identifying potential gender-based inequality in Aadhaar access.

**How**: This requires a 'female_enrolments' column, which is not in the current dataset. We will simulate this by assuming a fixed percentage and demonstrate the logic.

In [7]:
# --- 5. Female Access Gap ---
print("\\n--- Insight 5: Female Access Gap ---")

if 'female_enrolments' not in enrolment_df.columns:
    print("[INFO] 'female_enrolments' column not found. Simulating data for demonstration.")
    # Simulate female enrolments as a random percentage (e.g., 35-50%) of total enrolments
    enrolment_df['female_enrolments'] = (enrolment_df['total_enrolment'] * np.random.uniform(0.35, 0.50, size=len(enrolment_df))).astype(int)

# Calculate female ratio per district
district_gender_totals = enrolment_df.groupby(['state', 'district']).agg(
    total_enrolments=('total_enrolment', 'sum'),
    female_enrolments=('female_enrolments', 'sum')
).reset_index()

district_gender_totals['female_ratio'] = district_gender_totals['female_enrolments'] / (district_gender_totals['total_enrolments'] + 1)

# Get state average female ratio
state_avg_female_ratio = district_gender_totals.groupby('state')['female_ratio'].mean().reset_index()
state_avg_female_ratio.rename(columns={'female_ratio': 'state_avg_female_ratio'}, inplace=True)

# Find districts with a significant gap
access_gap_districts = pd.merge(district_gender_totals, state_avg_female_ratio, on='state')
access_gap_districts = access_gap_districts[access_gap_districts['female_ratio'] < 0.8 * access_gap_districts['state_avg_female_ratio']] # Gap is >20% below state avg

print(f"Found {len(access_gap_districts)} districts with a potential female access gap.")
display(access_gap_districts.sort_values('female_ratio').head(10))

\n--- Insight 5: Female Access Gap ---
[INFO] 'female_enrolments' column not found. Simulating data for demonstration.
Found 123 districts with a potential female access gap.


,state,district,total_enrolments,female_enrolments,female_ratio,state_avg_female_ratio
0,Andaman And Nicobar Islands,Nicobars,1,0,0.0,0.022556
658,Uttar Pradesh,Mahrajganj,19,0,0.0,0.347561
521,Rajasthan,Salumbar,1,0,0.0,0.291997
497,Rajasthan,Beawar,1,0,0.0,0.291997
493,Rajasthan,Balotra,1,0,0.0,0.291997
416,Mizoram,Hnahthial,2,0,0.0,0.199454
213,Himachal Pradesh,Lahaul And Spiti,3,0,0.0,0.104815
50,Assam,Bajali,29,0,0.0,0.323922
40,Arunachal Pradesh,Pakke Kessang,4,0,0.0,0.149979
33,Arunachal Pradesh,Leparada,3,0,0.0,0.149979


### Insight 6: Elderly Update Stress Zones
**What**: Identifying areas where elderly citizens (age 60+) require a disproportionately high number of updates, suggesting usability or accessibility issues.

**How**: This requires age-specific update data, which is not fully present. We will simulate the logic using available columns.

In [8]:
# --- 6. Elderly Update Stress Zones ---
print("\\n--- Insight 6: Elderly Update Stress Zones ---")

# The dataset has 'age_18_greater' and 'bio_age_17_'. We'll use these as a proxy for the adult/elderly population.
# A real implementation would need a specific 'age_60_plus' column.
print("[INFO] 'elderly' columns not found. Using 'age_18_greater' and 'bio_age_17_' as proxies.")

# Proxy for elderly enrolments and updates
enrolment_df['elderly_enrolments_proxy'] = enrolment_df['age_18_greater']
biometric_df['elderly_updates_proxy'] = biometric_df['bio_age_17_']
demographic_df['elderly_updates_proxy'] = demographic_df['demo_age_17_']

# Aggregate at district level
elderly_enrol = enrolment_df.groupby(['state', 'district'])['elderly_enrolments_proxy'].sum().reset_index()
elderly_bio = biometric_df.groupby(['state', 'district'])['elderly_updates_proxy'].sum().reset_index()
elderly_demo = demographic_df.groupby(['state', 'district'])['elderly_updates_proxy'].sum().reset_index()

# Merge
elderly_stats = pd.merge(elderly_enrol, pd.merge(elderly_bio, elderly_demo, on=['state', 'district'], how='outer'), on=['state', 'district'], how='outer').fillna(0)
elderly_stats['total_elderly_updates'] = elderly_stats['elderly_updates_proxy_x'] + elderly_stats['elderly_updates_proxy_y']

# Calculate ratio
elderly_stats['elderly_update_ratio'] = elderly_stats['total_elderly_updates'] / (elderly_stats['elderly_enrolments_proxy'] + 1)

# Find high-stress zones (e.g., top 10%)
stress_threshold = elderly_stats['elderly_update_ratio'].quantile(0.90)
elderly_stress_zones = elderly_stats[elderly_stats['elderly_update_ratio'] > stress_threshold]

print(f"Found {len(elderly_stress_zones)} potential elderly update stress zones (top 10%).")
display(elderly_stress_zones.sort_values('elderly_update_ratio', ascending=False).head(10))

\n--- Insight 6: Elderly Update Stress Zones ---
[INFO] 'elderly' columns not found. Using 'age_18_greater' and 'bio_age_17_' as proxies.
Found 72 potential elderly update stress zones (top 10%).


,state,district,elderly_enrolments_proxy,elderly_updates_proxy_x,elderly_updates_proxy_y,total_elderly_updates,elderly_update_ratio
598,Telangana,Warangal,0.0,75317.0,54786.0,130103.0,130103.000000
194,Haryana,Jind,0.0,56391.0,39090.0,95481.0,95481.000000
546,Tamil Nadu,Nagapattinam,0.0,45373.0,34068.0,79441.0,79441.000000
562,Tamil Nadu,Tirunelveli,2.0,117524.0,65176.0,182700.0,60900.000000
548,Tamil Nadu,Perambalur,0.0,31894.0,12354.0,44248.0,44248.000000
567,Tamil Nadu,Viluppuram,3.0,98424.0,63119.0,161543.0,40385.750000
135,Chhattisgarh,Mungeli,2.0,53653.0,66729.0,120382.0,40127.333333
541,Tamil Nadu,Kanniyakumari,1.0,41917.0,32451.0,74368.0,37184.000000
130,Chhattisgarh,Jashpur,2.0,44353.0,62323.0,106676.0,35558.666667
205,Haryana,Sirsa,2.0,56424.0,50186.0,106610.0,35536.666667


### Insight 7: Migration Magnet Districts
**What**: Districts that appear to be absorbing migrants, characterized by high update volumes but relatively low new enrolments.

**How**: We identify districts with a high `update_ratio` (from Insight 2) and a low `first_time_ratio` (from Insight 4), which signals a population that is already enrolled but relocating.

In [9]:
# --- 7. Migration Magnet Districts ---
print("\\n--- Insight 7: Migration Magnet Districts ---")

# We can reuse the ratios calculated in previous insights
migration_analysis = district_totals.copy()

# Define thresholds
high_update_threshold = migration_analysis['update_ratio'].quantile(0.75) # Top 25% in updates
low_enrolment_threshold = migration_analysis['first_time_ratio'].quantile(0.25) # Bottom 25% in new enrolments

migration_magnets = migration_analysis[
    (migration_analysis['update_ratio'] > high_update_threshold) &
    (migration_analysis['first_time_ratio'] < low_enrolment_threshold)
]

print(f"Found {len(migration_magnets)} potential migration magnet districts.")
display(migration_magnets.sort_values('update_ratio', ascending=False).head(10))

\n--- Insight 7: Migration Magnet Districts ---
Found 177 potential migration magnet districts.


,state,district,total_enrolments,total_updates,update_ratio,first_time_ratio
421,Mizoram,Mamit,27.0,11209.0,400.321429,0.002403
599,The Dadra And Nagar Haveli And Daman And Diu,Daman,31.0,6646.0,207.687500,0.004642
400,Manipur,Thoubal,370.0,75226.0,202.765499,0.004894
121,Chhattisgarh,Balod,530.0,107142.0,201.774011,0.004922
393,Manipur,Imphal East,366.0,71928.0,195.989101,0.005063
387,Maharashtra,Wardha,569.0,108865.0,190.991228,0.005199
139,Chhattisgarh,Rajnandgaon,1262.0,238189.0,188.589865,0.005270
423,Mizoram,Serchhip,49.0,9183.0,183.660000,0.005307
20,Andhra Pradesh,Srikakulam,1430.0,254888.0,178.118798,0.005579
134,Chhattisgarh,Mahasamund,1077.0,191978.0,178.087199,0.005579


### Insight 8: Seasonal Enrolment Elasticity
**What**: Measuring how sensitive a district's enrolment activities are to seasonal changes.

**How**: We calculate the coefficient of variation (Std Dev / Mean) for monthly enrolment totals in each district. A high value indicates significant seasonal fluctuations.

In [10]:
# --- 8. Seasonal Enrolment Elasticity ---
print("\\n--- Insight 8: Seasonal Enrolment Elasticity ---")

master_df['year_month'] = master_df['date'].dt.to_period('M')

# Group by month and district
monthly_district_enrolment = master_df.groupby(['state', 'district', 'year_month'])['total_enrolment'].sum().reset_index()

# Calculate coefficient of variation for each district
seasonality = monthly_district_enrolment.groupby(['state', 'district'])['total_enrolment'].agg(['mean', 'std']).reset_index()
seasonality['coeff_of_variation'] = seasonality['std'] / (seasonality['mean'] + 1)

# Identify highly seasonal districts (e.g., top 10%)
seasonality_threshold = seasonality['coeff_of_variation'].quantile(0.90)
highly_seasonal_districts = seasonality[seasonality['coeff_of_variation'] > seasonality_threshold]

print(f"Found {len(highly_seasonal_districts)} districts with high seasonal elasticity (top 10%).")
display(highly_seasonal_districts.sort_values('coeff_of_variation', ascending=False).head(10))

\n--- Insight 8: Seasonal Enrolment Elasticity ---
Found 72 districts with high seasonal elasticity (top 10%).


,state,district,mean,std,coeff_of_variation
106,Bihar,Pashchim Champaran,1348.083333,4661.080278,3.454998
512,Rajasthan,Jalore,40.916667,141.424865,3.373953
182,Gujarat,Surendranagar,112.916667,377.628355,3.314953
146,Delhi,New Delhi,125.833333,419.534121,3.307759
413,Meghalaya,West Khasi Hills,1150.333333,3731.953481,3.241419
412,Meghalaya,West Jaintia Hills,830.500000,2663.846416,3.203664
411,Meghalaya,West Garo Hills,1161.083333,3691.679452,3.176777
403,Meghalaya,East Jaintia Hills,384.083333,1218.856354,3.165176
62,Assam,Dima Hasao,18.916667,63.014368,3.163901
433,Nagaland,Peren,107.083333,341.058901,3.155518


### Insight 9: Administrative Friction Hotspots
**What**: Places where Aadhaar corrections may be repeatedly failing, indicated by high update volumes but low daily throughput.

**How**: We identify districts with a high total number of updates but a low average number of updates per active day.

In [11]:
# --- 9. Administrative Friction Hotspots ---
print("\\n--- Insight 9: Administrative Friction Hotspots ---")

# Calculate total updates and active update days per district
district_update_activity = master_df[master_df['total_updates'] > 0].groupby(['state', 'district']).agg(
    total_updates=('total_updates', 'sum'),
    active_update_days=('date', 'nunique')
).reset_index()

district_update_activity['avg_updates_per_day'] = district_update_activity['total_updates'] / district_update_activity['active_update_days']

# Define thresholds
high_updates_threshold = district_update_activity['total_updates'].quantile(0.75) # Top 25% in update volume
low_throughput_threshold = district_update_activity['avg_updates_per_day'].quantile(0.25) # Bottom 25% in daily efficiency

friction_hotspots = district_update_activity[
    (district_update_activity['total_updates'] > high_updates_threshold) &
    (district_update_activity['avg_updates_per_day'] < low_throughput_threshold)
]

print(f"Found {len(friction_hotspots)} potential administrative friction hotspots.")
display(friction_hotspots.sort_values('avg_updates_per_day').head(10))

\n--- Insight 9: Administrative Friction Hotspots ---
Found 0 potential administrative friction hotspots.


,state,district,total_updates,active_update_days,avg_updates_per_day


### Insight 10: Data Quality Confidence Map (Meta Insight)
**What**: A map showing where the cleaning pipeline dropped the most rows, indicating areas with poorer initial data quality.

**How**: We read the log files generated by the cleaning pipeline, count the dropped rows for each source file, and aggregate them by state and district.

In [12]:
# --- 10. Data Quality Confidence Map ---
print("\\n--- Insight 10: Data Quality Confidence Map ---")

def load_log_files(logs_dir: Path) -> pd.DataFrame:
    log_files = list(logs_dir.glob("*_unresolved.csv"))
    if not log_files:
        print("[WARN] No log files found.")
        return pd.DataFrame()
    
    log_dfs = [pd.read_csv(file) for file in log_files]
    return pd.concat(log_dfs, ignore_index=True)

dropped_rows_df = load_log_files(LOGS_DIR)

if not dropped_rows_df.empty:
    # The district and state in the logs are the *raw* values. We'll group by them.
    quality_map = dropped_rows_df.groupby(['raw_state', 'raw_district']).size().reset_index(name='dropped_rows_count')
    
    # For context, let's get the original total rows (approximate)
    # This requires reading the raw data, which can be slow. We'll show the top dropped districts instead.
    
    print(f"Total rows dropped by LGD validation: {len(dropped_rows_df):,}")
    print("Top 15 locations with the most dropped rows (potential data quality issues):")
    display(quality_map.sort_values('dropped_rows_count', ascending=False).head(15))
else:
    print("No dropped row logs were found, data quality appears to be high or logs are missing.")

\n--- Insight 10: Data Quality Confidence Map ---
Total rows dropped by LGD validation: 701,669
Top 15 locations with the most dropped rows (potential data quality issues):


,raw_state,raw_district,dropped_rows_count
228,West Bengal,Barddhaman,28256
95,Karnataka,Bengaluru,23752
92,Karnataka,Bangalore,19764
94,Karnataka,Belgaum,18905
126,Maharashtra,Ahmadnagar,18831
3,Andhra Pradesh,Anantapur,16821
188,Tamil Nadu,Tiruvallur,16061
5,Andhra Pradesh,Cuddapah,15751
9,Andhra Pradesh,Nellore,15520
113,Kerala,Pathanamthitta,15051
